# Pilot TR EDA
**Scope:** Fire season timing metrics (onset, peak, end, season length) across 13–14 WWF RESOLVE ecoregions in Turkey, 2003 - 2025, using combined Terra + Aqua MODIS active fire detections.

**Outputs:** Graphs saved to `outputs/turkey_ecoregions/graphs/`

In [8]:
# IMPORTS & LOAD -----------------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns
from scipy import stats
import glob

# Load the assembled master CSV — one row per ecoregion per year.
# This file is produced by the pipeline notebook (00_Pilot_TR_pipeline.ipynb)
# by globbing all per-ecoregion CSVs from outputs/turkey_ecoregions/.
output_dir = 'outputs/turkey_ecoregions'
master_df  = pd.read_csv(f'{output_dir}/master_turkey.csv')

# Output directory
graphs_dir = 'outputs/turkey_ecoregions/graphs'
os.makedirs(graphs_dir, exist_ok=True)

# Reporting prints
print(f'Loaded {len(master_df)} rows')
print(f'Ecoregions: {master_df["eco_name"].nunique()}')
print(f'Years: {master_df["year"].min()} – {master_df["year"].max()}')
print()
print(master_df.head())

Loaded 304 rows
Ecoregions: 14
Years: 2003 – 2025

   eco_id                                       eco_name  year  onset_doy  \
0     786  Anatolian conifer and deciduous mixed forests  2003         94   
1     786  Anatolian conifer and deciduous mixed forests  2004         81   
2     786  Anatolian conifer and deciduous mixed forests  2005         74   
3     786  Anatolian conifer and deciduous mixed forests  2006         86   
4     786  Anatolian conifer and deciduous mixed forests  2007         82   

   peak_doy  end_doy  season_length  n_detections  
0       207      295            202           878  
1       222      308            228           971  
2       196      312            239           804  
3       192      281            196           707  
4       206      283            202           969  


## Data Filtering

Remove ecoregions with fewer than `MIN_VALID_YEARS` valid years.



In [ ]:
# FILTER: EXCLUDE ECOREGIONS WITH FEWER THAN 20 VALID YEARS ----------------------------------------
# Years where pipeline failed are simply absent from master_df (no row written).
# So counting rows per ecoregion = counting valid years directly.

MIN_VALID_YEARS = 20

valid_year_counts = (
    master_df
    .groupby('eco_name')['year']
    .count()
    .rename('n_valid_years')
)

# Split into passing and excluded
passing  = valid_year_counts[valid_year_counts >= MIN_VALID_YEARS].index
excluded = valid_year_counts[valid_year_counts <  MIN_VALID_YEARS]

if len(excluded) > 0:
    print(f'Excluded {len(excluded)} ecoregion(s) with < {MIN_VALID_YEARS} valid years:')
    for name, n in excluded.items():
        print(f'  {name}: {n} valid years')
else:
    print('No ecoregions excluded.')

# Apply filter
master_df = master_df[master_df['eco_name'].isin(passing)].copy()

print(f'\nRetained: {master_df["eco_name"].nunique()} ecoregions')
print(f'Rows:     {len(master_df)}')

# Section 1: General EDA

## Graph 01: Per-Ecoregion Time Series


In [ ]:
# GRAPH 01: Per-Ecoregion Time Series --------------------------------------------------------------

fig, axes = plt.subplots(n_ecos, 1, figsize=(14, n_ecos * 2.5))
fig.suptitle('Fire Season Timing: Turkey Ecoregions (2003–2025)\nSeason Stems + Peak',
             fontsize=14, y=1.01)

for row_idx, eco_name in enumerate(sorted(eco_names)):
    eco_df = master_df[master_df['eco_name'] == eco_name].sort_values('year')
    ax = axes[row_idx]

    ax.plot(eco_df['year'], eco_df['onset_doy'],
        color='orange', linewidth=1, alpha=0.5, zorder=2)
    ax.plot(eco_df['year'], eco_df['end_doy'],
        color='steelblue', linewidth=1, alpha=0.5, zorder=2)
    ax.plot(eco_df['year'], eco_df['peak_doy'],
        color='firebrick', linewidth=1, alpha=0.5, zorder=2)
    
    ax.vlines(eco_df['year'], eco_df['onset_doy'], eco_df['end_doy'],
              color='black', linewidth=1, label='Season lenght')
    ax.scatter(eco_df['year'], eco_df['onset_doy'],
               color='orange', zorder=3, s=30, label='Onset')
    ax.scatter(eco_df['year'], eco_df['end_doy'],
               color='steelblue', zorder=3, s=30, label='End')
    ax.scatter(eco_df['year'], eco_df['peak_doy'],
               color='firebrick', zorder=4, s=30, marker='D', label='Peak')

    ax.set_yticks([90, 180, 270, 360])
    ax.set_yticks([45, 135, 225, 315], minor=True)
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.grid(True, alpha=0.3)
    ax.grid(True, which='minor', alpha=0.15)
    ax.set_xticks(eco_df['year'])
    ax.tick_params(axis='x', rotation=90, labelsize=7)

    ax.set_ylabel(eco_name.replace(' ', '\n'), fontsize=9,
                  rotation=0, labelpad=10)
    ax.yaxis.label.set_ha('right')
    ax.yaxis.label.set_va('center')

    if row_idx == 0:
        ax.legend(fontsize=7, loc='upper left', ncol=2)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_01_FST_lollipop.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(n_ecos, 2, figsize=(14, n_ecos * 2.5))
fig.suptitle('Fire Season Timing: Turkey Ecoregions (2003 - 2025)', fontsize=14, y=1.01)

timing_metrics = [
    ('onset_doy',  'Onset',    'orange'),
    ('peak_doy',   'Peak',     'firebrick'),
    ('end_doy',    'End',      'steelblue'),
]

for row_idx, eco_name in enumerate(sorted(eco_names)):
    eco_df = master_df[master_df['eco_name'] == eco_name].sort_values('year')

    # --- Left plot: onset + peak + end together ---
    ax_timing = axes[row_idx, 0]
    for metric, label, color in timing_metrics:
        ax_timing.plot(eco_df['year'], eco_df[metric],
                       marker='o', color=color, linewidth=1.5,
                       markersize=4, label=label)

    ax_timing.set_yticks([90, 180, 270, 360], minor=False)
    ax_timing.set_yticks([45, 135, 225, 315], minor=True)
    ax_timing.yaxis.set_minor_formatter(plt.NullFormatter())
    ax_timing.grid(True, alpha=0.3)
    ax_timing.grid(True, which='minor', alpha=0.15)
    ax_timing.set_xticks(eco_df['year'])
    ax_timing.tick_params(axis='x', rotation=90, labelsize=7)

    if row_idx == 0:
        ax_timing.set_title('Timing (DOY)', fontsize=10)
        ax_timing.legend(fontsize=7, loc='upper left')

    if col_idx == 0:
        ax_timing.set_ylabel(eco_name.replace(' ', '\n'), fontsize=10,
                             rotation=0, labelpad=10)
        ax_timing.yaxis.label.set_ha('right')
        ax_timing.yaxis.label.set_va('center')

    # --- Right plot: season length ---
    ax_len = axes[row_idx, 1]
    ax_len.plot(eco_df['year'], eco_df['season_length'],
                marker='o', color='black', linewidth=1.5, markersize=4)
    ax_len.grid(True, alpha=0.3)
    ax_len.set_xticks(eco_df['year'])
    ax_len.tick_params(axis='x', rotation=90, labelsize=7)

    if row_idx == 0:
        ax_len.set_title('Season Length (days)', fontsize=10)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_01v2_FST.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Graph 02: Heatmaps

In [ ]:
# GRAPH 02: Heatmaps -------------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Fire Season Timing Heatmaps — Turkey Ecoregions (2003 - 2025)', fontsize=13)

metrics = [
    ('onset_doy',     'Onset DOY',            'RdYlBu_r'),
    ('peak_doy',      'Peak DOY (centroid)',   'RdYlBu_r'),
    ('end_doy',       'End DOY',              'RdYlBu_r'),
    ('season_length', 'Season Length (days)', 'pink_r'),
]

for ax, (metric, title, cmap) in zip(axes.flatten(), metrics):
    pivot = master_df.pivot(index='eco_name', columns='year', values=metric)

    # Centre diverging colormaps on the cross-ecoregion mean.
    # Season length is not centred — there is no meaningful midpoint.
    center = np.nanmean(pivot.values) if metric != 'season_length' else None

    sns.heatmap(
        pivot,
        ax         = ax,
        cmap       = cmap,
        center     = center,
        linewidths = 0.4,
        linecolor  = 'white',
        annot      = False,
        cbar_kws   = {'shrink': 0.8, 'label': metric}
    )

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=8)
    ax.tick_params(axis='x', labelsize=8, rotation=90)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_02_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Graph 03: Trends

**Question:** Are fire season metrics shifting over the 2003 - 2025 period across Turkey ecoregions?

**What the trends show:**
- **Onset (p=0.01, slope=−1.81 days/yr):** Strongest result. ~40 days earlier onset over 22 years. Direction likely real but magnitude uncertain: sparse-detection years in recent years may be pulling the cross-ecoregion mean artificially low.
- **Peak (p=0.92, slope=−0.05/yr):** No trend. Meaningful null result. the fire intensity centroid is stable even as onset shifts. The 2008–2012 volatility in the mean warrants investigation (linked to the suspicious Central Anatolian steppe cell in the heatmap).
- **End (p=0.02, slope=+0.69/yr):** Modest later-end trend (~15 days over 22 years). Sensitive to outliers. One ecoregion with very late sparse detections in recent years may be inflating the slope.
- **Season length (p=0.00, slope=+2.50/yr):** Strongest p-value but most suspect. Season length inherits noise from both onset and end. Values exceeding 300 days in some ecoregion-years are implausible as real fire seasons. Trend is likely partially real but inflated by metric artefacts.

**Before interpreting these trends as final results:**
- Identify ecoregion-years with total detections below a minimum threshold
- Recompute trends after excluding flagged years
- Check whether slopes and p-values hold up after cleaning



In [ ]:
# GRAPH 03: Trends ---------------------------------------------------------------------------------
# OLS trend is fitted to the cross-ecoregion yearly mean, not to individual ecoregions.
# The 95% CI around the trend line is computed analytically from OLS standard error,
# not from bootstrapping — it reflects uncertainty in the slope estimate only.

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Fire Season Timing Trends — Turkey Ecoregions (2003 - 2025)', fontsize=13)

metrics = [
    ('onset_doy',     'Onset DOY',            'orange'),
    ('peak_doy',      'Peak DOY (centroid)',   'firebrick'),
    ('end_doy',       'End DOY',              'steelblue'),
    ('season_length', 'Season Length (days)', 'green'),
]

for ax, (metric, title, color) in zip(axes.flatten(), metrics):

    # Faint individual ecoregion lines for context
    for eco_name, group in master_df.groupby('eco_name'):
        ax.plot(group['year'], group[metric],
                color=color, alpha=0.2, linewidth=1)

    # Cross-ecoregion mean and ±1 SD band
    yearly_mean = master_df.groupby('year')[metric].mean()
    yearly_std  = master_df.groupby('year')[metric].std()
    years       = yearly_mean.index

    ax.plot(years, yearly_mean, color=color, linewidth=2.5, label='Mean')
    ax.fill_between(years,
                    yearly_mean - yearly_std,
                    yearly_mean + yearly_std,
                    color=color, alpha=0.15, label='±1 SD')

    # OLS trend line fitted to the yearly mean
    slope, intercept, r, p, se = stats.linregress(years, yearly_mean)
    trend = slope * np.array(years) + intercept

    # Analytical 95% CI around the trend line
    # ci widens away from the centre of the x-range (standard OLS behaviour)
    n      = len(years)
    t_crit = stats.t.ppf(0.975, df=n - 2)
    x_mean = np.mean(years)
    ss_x   = np.sum((np.array(years) - x_mean) ** 2)
    ci     = t_crit * se * np.sqrt(1/n + (np.array(years) - x_mean)**2 / ss_x)

    ax.plot(years, trend, color='black', linewidth=1.5, linestyle='--',
            label=f'Trend (p={p:.2f}, slope={slope:.2f}/yr)')
    ax.fill_between(years, trend - ci, trend + ci,
                    color='black', alpha=0.1, label='95% CI')

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Year')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_03_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## Graph 04: Spatial Map
**Data:** Ecoregion geometries are loaded from the pre-saved `eco_geometries.json` (produced by the pipeline notebook).


In [ ]:
# LOAD CLIPPED GEOMETRIES --------------------------------------------------------------------------

import json
import geopandas as gpd
from shapely.geometry import shape

with open('outputs/turkey_ecoregions/eco_geometries.json') as f:
    geo_records = json.load(f)

eco_records = [
    {'eco_id': r['eco_id'], 'eco_name': r['eco_name'], 'geometry': shape(r['geometry'])}
    for r in geo_records
]

print(f'Loaded {len(eco_records)} clipped ecoregion geometries.')

In [ ]:
# GRAPH 04: Spatial Map ----------------------------------------------------------------------------

master_df = pd.read_csv('outputs/turkey_ecoregions/master_turkey.csv')

# Mean of each metric across all valid years per ecoregion
mean_metrics = master_df.groupby('eco_name')[
    ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
].mean().reset_index()

# Build GeoDataFrame and merge mean metrics
eco_gdf    = gpd.GeoDataFrame(eco_records, crs='EPSG:4326')
eco_merged = eco_gdf.merge(mean_metrics, on='eco_name')

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Mean Fire Season Timing — Turkey Ecoregions', fontsize=13)

metrics = [
    ('onset_doy',     'Mean Onset DOY',           'RdYlBu_r'),
    ('peak_doy',      'Mean Peak DOY (centroid)',  'RdYlBu_r'),
    ('end_doy',       'Mean End DOY',              'RdYlBu_r'),
    ('season_length', 'Mean Season Length (days)', 'RdYlBu_r'),
]

for ax, (metric, title, cmap) in zip(axes.flatten(), metrics):
    eco_merged.plot(column=metric, ax=ax, cmap=cmap,
                    legend=True, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_04_map.png', dpi=150, bbox_inches='tight')
plt.show()

## Graph 05: Metric Relationships

**Question:** How are the four timing metrics correlated with each other across all ecoregion-years?

**Distributions (diagonal):**
- **Onset DOY:** Bimodal: two clusters around DOY ~90 and ~175. The early cluster likely reflects Mediterranean/Aegean ecoregions, the late cluster continental/eastern ones. Worth investigating whether these two groups have different climate drivers.
- **Peak DOY:** Strongly unimodal, tightly concentrated around DOY ~220–230 (August). The most well-behaved distribution of the four confirms peak is the most stable and reliable metric.
- **End DOY:** Unimodal but right-skewed most ecoregion-years end around DOY ~300–310, with a tail extending to DOY ~360. The tail is likely artefact-driven (sparse late detections extending the computed end).
- **Season length:** Left-skewed with a long tail toward 300+ days. The bulk sits around 180–250 days, which is plausible. Values above ~300 days are implausible for a real fire season and are candidates for exclusion.

**Key correlations (off-diagonal):**
- **onset vs season_length (r=−0.94):** The dominant relationship in the dataset earlier onset almost perfectly predicts longer season. The scatter is remarkably tight and linear. Since end DOY is relatively stable, onset essentially determines season length. This has a direct implication for RQ1: explaining onset is largely equivalent to explaining season length.
- **onset vs peak (r=0.48):** Moderate positive correlation earlier-starting seasons tend to peak earlier too, but the relationship is loose.
- **end vs season_length (r=0.43):** Later end also contributes to longer seasons but much less strongly than onset does. End varies more independently.
- **onset vs end (r=−0.10):** Near-zero onset and end are essentially independent. Early-starting seasons do not systematically end later. This means the season length signal is almost entirely driven by onset variation not end variation.
- **peak vs end (r=0.26):** Weak, later-peaking seasons tend to end slightly later, which is logical but not a strong constraint.
- **peak vs season_length (r=−0.34):** Moderate negative, earlier peak associated with longer season, consistent with the onset relationship since onset and peak are correlated.

**Key implication for RQ1–RQ3:**
- Onset is the metric to focus on. It has the strongest climate sensitivity candidates, drives season length almost entirely, and shows the clearest spatial and temporal structure.
- Peak is the cleanest metric for cross-ecoregion comparison due to its unimodal, low-variance distribution, useful as a secondary metric in RQ2 predictability analysis.
- Season length is largely redundant with onset given r=−0.94, but may still be useful as a directly policy-relevant metric.
- The bimodal onset distribution suggests Turkey's ecoregions may naturally split into two fire regime groups. this is the seed of the typology in RQ3.

In [ ]:
# ── GRAPH 05: Scatter Matrix ──────────────────────────────────────────────────
# Each point = one ecoregion-year observation (n ≈ 13 ecoregions × 22 years = ~286 points).
# Pearson r annotated in each off-diagonal panel.
# Note: r values here are across all ecoregion-years pooled, not per-ecoregion.
# Between-ecoregion differences (e.g. one ecoregion always having late onset)
# contribute to the correlations — this is expected and appropriate for the
# cross-sectional analysis in RQ3/RQ4.

from pandas.plotting import scatter_matrix

cols = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Metric Relationships — Turkey Ecoregions (2003 - 2025)', fontsize=13)

scatter_matrix(
    master_df[cols],
    ax        = axes,
    alpha     = 0.3,
    diagonal  = 'hist',
    color     = 'steelblue',
    hist_kwds = {'bins': 15, 'color': 'steelblue', 'alpha': 0.7}
)

# Annotate each off-diagonal panel with Pearson r
for i, col1 in enumerate(cols):
    for j, col2 in enumerate(cols):
        if i != j:
            r, p = stats.pearsonr(
                master_df[col1].dropna(),
                master_df[col2].dropna()
            )
            axes[i, j].annotate(f'r={r:.2f}', xy=(0.05, 0.88),
                                 xycoords='axes fraction', fontsize=8,
                                 color='firebrick')

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_05_scatter_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Graph 06: Activity Profiles


In [ ]:
# GRAPH 06: Setup 
DAILY_DIR    = f'{output_dir}/daily_counts'
SMOOTH_WIN   = 7
NCOLS        = 3
MONTH_TICKS  = [15, 46, 75, 106, 136, 167, 197, 228, 259, 289, 320, 350]
MONTH_LABELS = ["J","F","M","A","M","J","J","A","S","O","N","D"]

# Load all daily CSVs
daily_files = sorted(glob.glob(f'{DAILY_DIR}/*_daily.csv'))
all_daily   = pd.concat([pd.read_csv(f) for f in daily_files], ignore_index=True)
all_daily   = all_daily.sort_values(['eco_name', 'year', 'doy']).reset_index(drop=True)

# Only ecoregions that passed the MIN_VALID_YEARS filter
ecoregions  = sorted(all_daily[all_daily['eco_name'].isin(master_df['eco_name'].unique())]['eco_name'].unique())
n_eco       = len(ecoregions)

# Mean markers from master_df
mean_markers = master_df.groupby('eco_name')[['onset_doy', 'peak_doy', 'end_doy']].mean()

print(f'Loaded {len(daily_files)} daily CSVs, {n_eco} ecoregions')

In [ ]:
# ── GRAPH 06a: Fire Activity Profiles — Mean + years + markers ────────────────
# Shows full-year fire activity profile per ecoregion.
# Ordered by total detections (highest first) so the most fire-active
# ecoregions appear first. 

# Order ecoregions by total detections, highest first
eco_totals = (
    all_daily[all_daily['eco_name'].isin(ecoregions)]
    .groupby('eco_name')['n_detections']
    .sum()
    .sort_values(ascending=False)
)
ecoregions_by_activity = eco_totals.index.tolist()

nrows = int(np.ceil(len(ecoregions_by_activity) / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 5, nrows * 3.2), sharex=True)

for i, eco in enumerate(ecoregions_by_activity):
    ax    = axes.flatten()[i]
    sub   = all_daily[all_daily['eco_name'] == eco]
    pivot = sub.pivot_table(index='doy', columns='year', values='n_detections', aggfunc='sum')
    pivot = pivot.reindex(range(1, 366)).fillna(0)

    # Total detections smoothed — one curve per year (faint)
    for yr in pivot.columns:
        yr_smooth = pivot[yr].rolling(SMOOTH_WIN, center=True, min_periods=1).mean()
        ax.plot(pivot.index, yr_smooth, color='steelblue', alpha=0.15, linewidth=0.6)

    # Total across all years smoothed — bold mean curve
    total_smooth = pivot.sum(axis=1).rolling(SMOOTH_WIN, center=True, min_periods=1).mean()
    ax.plot(pivot.index, total_smooth, color='steelblue', linewidth=2, label='Total')

    # Onset / peak / end markers
    if eco in mean_markers.index:
        m = mean_markers.loc[eco]
        ax.axvline(m['onset_doy'], color='orange',    linewidth=1.2, linestyle='--', label='Onset')
        ax.axvline(m['peak_doy'],  color='firebrick',  linewidth=1.2, linestyle='--', label='Peak')
        ax.axvline(m['end_doy'],   color='royalblue',  linewidth=1.2, linestyle='--', label='End')

    if i == 0:
        ax.legend(fontsize=6, loc='upper left')

    ax.set_title(f'{eco}  ({int(eco_totals[eco]):,} det.)', fontsize=7)
    ax.set_xticks(MONTH_TICKS)
    ax.set_xticklabels(MONTH_LABELS, fontsize=7)
    ax.set_xlim(1, 365)
    ax.set_ylabel('Total detections', fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes.flatten())):
    axes.flatten()[j].set_visible(False)

fig.suptitle('Fire Activity Profiles — Turkey Ecoregions (2003 - 2025)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_06_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# Section 2: Metric Verification

These plots test whether onset, peak, and end DOY are landing where the raw detection data says they should.

In [ ]:
# DAILY DATA SETUP — shared across all plots below notebook ----------------------------------------
# Loads all daily CSVs and computes shared lookup tables used by multiple plots.

DAILY_DIR   = f'{output_dir}/daily_counts'
SMOOTH_WIN  = 7      # rolling-mean smoothing window (days)
NCOLS       = 3      # subplot grid columns

MONTH_TICKS  = [15, 46, 75, 106, 136, 167, 197, 228, 259, 289, 320, 350]
MONTH_LABELS = ['J','F','M','A','M','J','J','A','S','O','N','D']

daily_files = sorted(glob.glob(f'{DAILY_DIR}/*_daily.csv'))
if not daily_files:
    raise FileNotFoundError(f'No daily CSVs found in {DAILY_DIR}')

all_daily = pd.concat([pd.read_csv(f) for f in daily_files], ignore_index=True)
all_daily = all_daily.sort_values(['eco_name', 'year', 'doy']).reset_index(drop=True)

# Only keep ecoregions that passed MIN_VALID_YEARS
ecoregions   = sorted(all_daily[all_daily['eco_name'].isin(master_df['eco_name'].unique())]['eco_name'].unique())
n_eco        = len(ecoregions)

# Mean onset/peak/end per ecoregion — used as markers in multiple plots
mean_markers = master_df.groupby('eco_name')[['onset_doy', 'peak_doy', 'end_doy']].mean()

print(f'Loaded {len(daily_files)} daily CSVs')
print(f'Ecoregions in daily data: {n_eco}')

### Graph 07: Cumulative Fire Activity Curves

**Logic:** For each ecoregion-year, cumulative % of annual detections is plotted vs DOY. Onset is defined as the DOY where this curve crosses 5% — this plot shows that threshold in action against the actual curves.

**What to look for:**
- The orange dashed line (mean onset) should sit near where curves lift off from near-zero. If it sits in the middle of a steep rise, the threshold is being triggered too late.
- Tightly bunched curves = consistent fire season timing across years. Wide spread = high interannual variability.
- Curves that rise very gradually (flat for long stretches) = sparse detection years where onset will be unstable.

**Red flags:**
- Onset line far to the left of where curves start rising = threshold triggered by isolated early detections (noise).
- Onset line in the middle or right half of the cumulative curve = onset is very late relative to actual fire activity.

In [ ]:
# ── GRAPH 07: Cumulative Fire Activity Curves ─────────────────────────────────
ONSET_THRESHOLD = 0.05   # must match the value used in your pipeline

nrows = int(np.ceil(n_eco / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 5, nrows * 3.2), sharex=True)

# Sort ecoregions by mean onset DOY for graph 07 — earliest onset first
ecoregions_by_onset = (
    mean_markers['onset_doy']
    .sort_values()
    .index
    .tolist()
)

for i, eco in enumerate(ecoregions_by_onset):
    ax = axes.flatten()[i]
    sub   = all_daily[all_daily['eco_name'] == eco]
    pivot = sub.pivot_table(index='doy', columns='year', values='n_detections', aggfunc='sum')
    pivot = pivot.reindex(range(1, 366)).fillna(0)

    # Individual year cumulative curves
    for yr in pivot.columns:
        total = pivot[yr].sum()
        if total == 0:
            continue
        cumulative = pivot[yr].cumsum() / total * 100
        ax.plot(pivot.index, cumulative, color='steelblue', alpha=0.15, linewidth=0.6)

    # Mean cumulative curve
    mean_series = pivot.mean(axis=1)
    total_mean  = mean_series.sum()
    if total_mean > 0:
        mean_cum = mean_series.cumsum() / total_mean * 100
        ax.plot(pivot.index, mean_cum, color='steelblue', linewidth=2, label='Mean')

    # Mean onset marker
    if eco in mean_markers.index:
        ax.axvline(mean_markers.loc[eco, 'onset_doy'], color='orange',
                   linewidth=1.5, linestyle='--', label='Mean onset')

    # Threshold reference line
    ax.axhline(ONSET_THRESHOLD * 100, color='gray', linewidth=0.8,
               linestyle=':', label=f'{int(ONSET_THRESHOLD*100)}% threshold')

    if i == 0:
        ax.legend(fontsize=6, loc='upper left')

    ax.set_title(eco, fontsize=8)
    ax.set_xticks(MONTH_TICKS)
    ax.set_xticklabels(MONTH_LABELS, fontsize=7)
    ax.set_xlim(1, 365)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Cumul. %', fontsize=7)
    ax.grid(True, alpha=0.3)

for j in range(i + 1, len(axes.flatten())):
    axes.flatten()[j].set_visible(False)

fig.suptitle('Cumulative Fire Activity Curves — Turkey Ecoregions (2003 - 2025)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_07_cumulative_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph 08: Coefficient of Variation per Ecoregion per Metric

**Logic:** Coefficient of variation (CV = std / mean × 100) normalises variability by the mean, making different metrics and ecoregions comparable on the same scale.

**What to look for:**
- High CV in onset but low CV in peak = the season start is erratic but the intensity peak is stable. This is ecologically meaningful and common in fire-prone systems.
- Ecoregions with high CV across all metrics = genuinely unpredictable fire regime, may need extra scrutiny before including in RQ2 predictability analysis.
- Season length CV should roughly reflect onset CV if end is stable — check if that holds.

**Red flags:**
- Very high CV in onset for ecoregions with sparse detections = likely noise, not real variability.

In [ ]:
# ── GRAPH 10: Coefficient of Variation per Ecoregion per Metric ───────────────
metrics_cv = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
colors_cv  = ['orange', 'firebrick', 'royalblue', 'green']
labels_cv  = ['Onset DOY', 'Peak DOY', 'End DOY', 'Season Length']

cv_df = (
    master_df.groupby('eco_name')[metrics_cv]
    .apply(lambda x: (x.std() / x.mean() * 100))
    .reset_index()
)

# Total detections per ecoregion — shown on bars as context for CV reliability
det_totals = (
    all_daily[all_daily['eco_name'].isin(ecoregions)]
    .groupby('eco_name')['n_detections']
    .sum()
)

cv_df['short_name'] = (cv_df['eco_name']
                       .str.replace(' forests', '')
                       .str.replace(' steppe', '')
                       .str.replace(' and ', ' & '))
cv_df['total_det']  = cv_df['eco_name'].map(det_totals)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Coefficient of Variation (%) per Ecoregion — Turkey Ecoregions',
             fontsize=13)

for ax, metric, color, label in zip(axes.flatten(), metrics_cv, colors_cv, labels_cv):
    sorted_cv = cv_df.sort_values(metric, ascending=False)
    bars = ax.bar(range(len(sorted_cv)), sorted_cv[metric],
                  color=color, alpha=0.75, edgecolor='white')

    # Annotate each bar with total detection count
    for bar, det in zip(bars, sorted_cv['total_det']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{int(det):,}',
            ha='center', va='bottom', fontsize=7, color='black', rotation=0
        )

    ax.set_xticks(range(len(sorted_cv)))
    ax.set_xticklabels(sorted_cv['short_name'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('CV (%)', fontsize=9)
    ax.set_title(label, fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    # Add extra headroom so rotated labels don't clip
    ax.set_ylim(0, sorted_cv[metric].max() * 1.35)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_10_cv_per_ecoregion.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph 09: Year Rankings — Earliest and Latest Years per Metric

**Logic:** For each metric, rank each year by its cross-ecoregion mean. Visualise the top 5 earliest and latest years as a simple bar chart. This gives an immediate answer to "which years were systematically anomalous".

**What to look for:**
- Do the same years appear as anomalous across multiple metrics? Consistent anomaly = likely climate-driven.
- Are the most recent years (2020–2025) overrepresented among the extremes? That would support the trend signal seen in Graph 03.
- The 2021 Türkiye fires — does 2021 rank as an anomalously late onset or long season?

**Red flags:**
- Same year appearing as both earliest and latest across different metrics = inconsistency worth investigating.

In [ ]:
# ── GRAPH 12: Year Rankings ───────────────────────────────────────────────────
N_EXTREMES = 5

metrics_rank = [
    ('onset_doy',     'Onset DOY',            'orange'),
    ('peak_doy',      'Peak DOY',             'firebrick'),
    ('end_doy',       'End DOY',              'royalblue'),
    ('season_length', 'Season Length (days)', 'green'),
]

yearly_means = master_df.groupby('year')[['onset_doy','peak_doy','end_doy','season_length']].mean()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Year Rankings — Earliest and Latest Years per Metric (cross-ecoregion mean)',
             fontsize=13)

for ax, (metric, title, color) in zip(axes.flatten(), metrics_rank):
    ranked = yearly_means[metric].sort_values()
    top5   = ranked.head(N_EXTREMES)    # earliest / shortest
    bot5   = ranked.tail(N_EXTREMES)    # latest / longest

    combined       = pd.concat([top5, bot5])
    bar_colors     = ['steelblue'] * N_EXTREMES + ['tomato'] * N_EXTREMES
    bar_labels     = [f'{yr}\n({val:.0f})' for yr, val in combined.items()]

    ax.bar(range(len(combined)), combined.values, color=bar_colors, alpha=0.8, edgecolor='white')
    ax.set_xticks(range(len(combined)))
    ax.set_xticklabels(bar_labels, fontsize=8)
    ax.axvline(N_EXTREMES - 0.5, color='black', linewidth=1, linestyle='--')
    ax.text(N_EXTREMES * 0.25 - 0.5, ax.get_ylim()[1] * 0.97,
            'Earliest/shortest', ha='center', fontsize=8, color='steelblue')
    ax.text(N_EXTREMES * 1.5 - 0.5, ax.get_ylim()[1] * 0.97,
            'Latest/longest', ha='center', fontsize=8, color='tomato')
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Mean DOY / days', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_12_year_rankings.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph 10: Metric vs Longitude and Latitude

**Logic:** Compute the centroid of each ecoregion from the saved geometries and plot each metric against longitude and latitude. Turkey has a strong west-east climate gradient (Mediterranean west, continental east) and a north-south gradient (Black Sea coast vs Anatolian plateau). These should show up in fire timing.

**What to look for:**
- Negative correlation of onset with longitude = western ecoregions start earlier (Mediterranean fire season starts in May/June vs July/August in the east).
- Season length vs latitude: southern ecoregions likely have longer seasons.

In [ ]:
# ── GRAPH 13: Metric vs Longitude and Latitude ────────────────────────────────
import json
from shapely.geometry import shape

with open(f'{output_dir}/eco_geometries.json') as f:
    geo_records = json.load(f)

# Compute centroids
centroids = []
for r in geo_records:
    geom = shape(r['geometry'])
    centroids.append({
        'eco_name': r['eco_name'],
        'lon':      geom.centroid.x,
        'lat':      geom.centroid.y,
    })
centroid_df = pd.DataFrame(centroids)

# Merge with mean metrics
mean_metrics = master_df.groupby('eco_name')[['onset_doy','peak_doy','end_doy','season_length']].mean().reset_index()
spatial_df   = centroid_df.merge(mean_metrics, on='eco_name')

metrics_sp = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
labels_sp  = ['Onset DOY', 'Peak DOY', 'End DOY', 'Season Length (days)']
colors_sp  = ['orange', 'firebrick', 'royalblue', 'green']

fig, axes = plt.subplots(4, 2, figsize=(12, 18))
fig.suptitle('Fire Season Metrics vs Ecoregion Centroid — Turkey', fontsize=13)

for row, (metric, label, color) in enumerate(zip(metrics_sp, labels_sp, colors_sp)):
    for col, coord in enumerate(['lon', 'lat']):
        ax = axes[row, col]
        ax.scatter(spatial_df[coord], spatial_df[metric],
                   color=color, s=60, alpha=0.8, edgecolors='white', zorder=3)

        # Add ecoregion name labels
        for _, row_data in spatial_df.iterrows():
            short = row_data['eco_name'].split()[-2] + ' ' + row_data['eco_name'].split()[-1]
            ax.annotate(short, (row_data[coord], row_data[metric]),
                        fontsize=5.5, ha='left', va='bottom', alpha=0.7)

        # Trend line
        slope, intercept, r, p, _ = stats.linregress(spatial_df[coord], spatial_df[metric])
        x_line = np.linspace(spatial_df[coord].min(), spatial_df[coord].max(), 100)
        ax.plot(x_line, slope * x_line + intercept,
                color='black', linewidth=1, linestyle='--',
                label=f'r={r:.2f}, p={p:.2f}')

        ax.set_xlabel('Longitude' if coord == 'lon' else 'Latitude', fontsize=9)
        ax.set_ylabel(label, fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_13_metric_vs_coordinates.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Section 3: Data Quality

These plots expose where data coverage is sparse enough to distrust the metrics. Identifying these cases now prevents them from contaminating the global analysis.

### Graph 11: Total Annual Detections Heatmap

In [ ]:
# GRAPH 14: Total Annual Detections Heatmap --------------------------------------------------------

# Sum detections per ecoregion per year from daily data
annual_counts = (
    all_daily[all_daily['eco_name'].isin(ecoregions)]
    .groupby(['eco_name', 'year'])['n_detections']
    .sum()
    .reset_index()
)

pivot_counts = annual_counts.pivot(index='eco_name', columns='year', values='n_detections')

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    pivot_counts,
    ax         = ax,
    cmap       = 'YlOrRd',
    linewidths = 0.4,
    linecolor  = 'white',
    annot      = True,
    fmt        = '.0f',
    annot_kws  = {'size': 6},
    cbar_kws   = {'shrink': 0.6, 'label': 'Total detections'}
)
ax.set_title('Total Annual Fire Detections per Ecoregion: Turkey (2003 - 2025)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Year', fontsize=9)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=8)
ax.tick_params(axis='x', labelsize=8, rotation=90)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_14_annual_detections_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()